# 🧠 Module 4: Agents & Tool Usage

---

## 🎯 Objective
Build an AI system that can decide when to use tools instead of answering directly.


## ⚙️ Setup (Reuse LLM)

In [1]:
import requests
import json

def ask_llm(prompt, model="mistral"):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        }
    )
    return response.json()["response"]

/Users/abhishekjha/Documents/ai-learning/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


#### 🧪 Step 1: Define Tools
---
👉 Tools = functions AI can call

In [2]:
# Tool 1: Calculator
def calculator(expression):
    """
    Evaluate a math expression safely
    Example: "2 + 2 * 5"
    """
    try:
        return str(eval(expression))
    except:
        return "Error in calculation"


# Tool 2: Knowledge base (mock DB)
def get_user_orders(user_id):
    """
    Fake database lookup
    """
    fake_db = {
        "user_1": "Last order: ₹500",
        "user_2": "Last order: ₹1200"
    }
    return fake_db.get(user_id, "No data found")

#### 🧪 Step 2: Tool Selection via Prompt
---
👉 We ask LLM to decide:
- Answer directly
- OR call a tool

In [3]:
def decide_tool(query):
    """
    Ask LLM which tool to use
    """
    
    prompt = f"""
    You are an AI assistant.

    Decide:
    - If math → use "calculator"
    - If user order → use "get_user_orders"
    - Otherwise → answer directly

    Return JSON:
    {{
        "action": "tool_name or answer",
        "input": "input for tool or final answer"
    }}

    Query:
    {query}
    """
    
    response = ask_llm(prompt)
    
    try:
        return json.loads(response)
    except:
        return {"action": "answer", "input": response}

#### 🧪 Step 3: Execute Tool

In [4]:
def execute_action(action_json):
    """
    Execute tool or return answer
    """
    
    action = action_json.get("action")
    tool_input = action_json.get("input")
    
    if action == "calculator":
        return calculator(tool_input)
    
    elif action == "get_user_orders":
        return get_user_orders(tool_input)
    
    else:
        return tool_input

#### 🧪 Step 4: Agent Loop

In [5]:
def agent(query):
    """
    Full agent pipeline:
    1. Decide action
    2. Execute tool
    """
    
    decision = decide_tool(query)
    
    print("Decision:", decision)  # debug
    
    result = execute_action(decision)
    
    return result

In [6]:
# Test agent
print(agent("What is 25 * 4?"))
print(agent("Show last order of user_1"))
print(agent("What is Python?"))

Decision: {'action': 'calculator', 'input': '25 * 4'}
100
Decision: {'action': 'get_user_orders', 'input': 'user_1'}
Last order: ₹500
Decision: {'action': 'answer', 'input': 'Python is a high-level, interpreted programming language used for various applications such as web development, artificial intelligence, data analysis, and scripting.'}
Python is a high-level, interpreted programming language used for various applications such as web development, artificial intelligence, data analysis, and scripting.


## 🏗 Mini Project: Smart Assistant

In [ ]:
while True:
    query = input("You: ")
    
    if query.lower() in ["exit", "quit"]:
        break
    
    response = agent(query)
    print("AI:", response)